In [ ]:
%pip install kagglehub
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install scikit-learn
%pip install plotly
%pip install kneed
%pip install jikan4snek

In [ ]:
import random
import numpy as np
import pandas as pd
import kagglehub as kb
import asyncio
import jikan4snek
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from typing import Counter
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [17]:
# params
top_n_anime = 15
top_n_clusters = 4
k_per_cluster = 5
most_popular_clusters = 20

In [18]:
# load the anime data set for the training
file_path = kb.dataset_download("CooperUnion/anime-recommendations-database")
print("Path to dataset files:", file_path)

Path to dataset files: C:\Users\MAXFRAME\.cache\kagglehub\datasets\CooperUnion\anime-recommendations-database\versions\1


In [19]:
# get the rating data
anime_data = pd.read_csv(f"{file_path}/anime.csv")
anime_data.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [20]:
# normalize the data
anime_data.loc[anime_data["episodes"] == "Unknown", "episodes"] = 0
# fill the rating empty cells
anime_data["rating"] = pd.to_numeric(anime_data["rating"], errors="coerce").fillna(0)
# create scalers
episodes_scaler = MinMaxScaler()
rating_scaler = MinMaxScaler()
members_scaler = MinMaxScaler()
# scale data
anime_data["episodes_norm"] = episodes_scaler.fit_transform(anime_data[["episodes"]])
anime_data["rating_norm"] = rating_scaler.fit_transform(anime_data[["rating"]])
anime_data["members_norm"] = members_scaler.fit_transform(anime_data[["members"]])

In [21]:
# create a list of all available genres
genre_mlb = MultiLabelBinarizer()
anime_data["genre_list"] = anime_data["genre"].fillna("").apply(lambda x: x.split(", "))
genre_encoded = genre_mlb.fit_transform(anime_data["genre_list"])
genre_df = pd.DataFrame(genre_encoded, columns=genre_mlb.classes_)

# let's do the same with types since i forgot these (0u0)
type_mlb = MultiLabelBinarizer()
type_encoded = type_mlb.fit_transform(anime_data["type"].fillna("").apply(lambda x: [x]))
type_df = pd.DataFrame(type_encoded, columns=type_mlb.classes_)

In [22]:
# construct the features df
feature_vectors = np.hstack([
    genre_df.values,
    type_df.values,
    anime_data[["episodes_norm", "rating_norm", "members_norm"]].values
])


In [23]:
# that check if any column have empty or NaN entries
anime_data[["episodes_norm", "rating_norm", "members_norm"]].isna().sum()

episodes_norm    0
rating_norm      0
members_norm     0
dtype: int64

In [24]:
# update i found kneed should found the optimal K automatically
inertia = []
ks = list(range(10, 201, 10))

# loop and search for the optimal k value
for k in ks:
    k_means = KMeans(n_clusters=k, random_state=42)
    k_means.fit(feature_vectors)
    inertia.append(k_means.inertia_)

# calculate optimal k
knee = KneeLocator(ks, inertia, curve="convex", direction="decreasing")
optimal_k = knee.knee

print(f"Optimal number of clusters: {optimal_k}")

Optimal number of clusters: 50


In [25]:
# create the k mean model
k_means = KMeans(n_clusters=optimal_k, random_state=42)
k_means.fit(feature_vectors)

,n_clusters,np.int64(50)
,init,'k-means++'
,n_init,'auto'
,max_iter,300
,tol,0.0001
,verbose,0
,random_state,42
,copy_x,True
,algorithm,'lloyd'


In [26]:
# so basically this group all anime and label them later you fetch all anime that share same cluster
anime_data["cluster"] = k_means.labels_

In [27]:
# get recommendations
def get_cluster_entries(anime_id):
  # get the cluster the anime belong to
  anime_index = anime_data[anime_data["anime_id"] == anime_id].index[0]
  target_cluster = anime_data.loc[anime_index, "cluster"]
  # get other elements excluding
  recommendations = anime_data[
    (anime_data["cluster"] == target_cluster) &
    (anime_data["anime_id"] != anime_id)  # exclude the original anime
  ]
  return recommendations

In [28]:
# get user matrixes
def get_user_vector(watched_ids_list, favorite_ids_list):
  # get the items idx
  watched_ids_list = anime_data[anime_data["anime_id"].isin(watched_ids_list)].index
  favorite_ids_list = anime_data[anime_data["anime_id"].isin(favorite_ids_list)].index

  # get the feature_vectors for each item
  user_vector = (
    feature_vectors[favorite_ids_list].mean(axis=0) * 2 +
    feature_vectors[watched_ids_list].mean(axis=0)
  ) / 3

  return user_vector.reshape(1,-1)


In [51]:
# convert the DataFrame to a list of anime IDs
def dataFrame_to_Ids(df):
    return df["anime_id"].tolist()


# basic anime loader to get the data from jikan API
async def get_anime_list(data_frame):
    # get the ids list 
    anime_list_ids = dataFrame_to_Ids(data_frame)
    # get the anime data from Jikan API
    Jikan = jikan4snek.Jikan4SNEK(debug=True)
    for i in anime_list_ids:
        res = await Jikan.get(i).anime()
    
    # display the anime title
    return len(res['data'])


In [40]:
# step two to level up that embarrassment of a system is to add cosine similarity
async def get_recommendations_based_on_entry(anime_id, watched_ids_list, favorite_ids_list, return_entry=False):
  # first we get all the same cluster entries
  same_cluster_entries = get_cluster_entries(anime_id)
  cluster_indices = same_cluster_entries.index
  cluster_vectors = feature_vectors[cluster_indices]

  # then we load the current anime data
  anime_idx = anime_data[anime_data["anime_id"] == anime_id].index[0]
  input_vector = feature_vectors[anime_idx].reshape(1, -1)

  # get the watched and favorite vector
  user_vector = get_user_vector(watched_ids_list, favorite_ids_list)
  final_vector = 0.3 * user_vector + 0.7 * input_vector


  # Compute cosine similarity
  similarities = cosine_similarity(final_vector, cluster_vectors).flatten()

  # rank the result and get top N items in the cosine similarity
  ranked_indices = cluster_indices[np.argsort(similarities)[::-1]]
  top_n_ids = anime_data.loc[ranked_indices, "anime_id"].values[:top_n_anime]

  # remove the items that are already in the user list or the current entry
  top_n_ids = [id for id in top_n_ids if id not in watched_ids_list + favorite_ids_list + [anime_id]]

  # get anime items for the ids
  top_n_items = anime_data[anime_data["anime_id"].isin(top_n_ids)]
  return await get_anime_list(top_n_items) if return_entry else top_n_items

In [ ]:
await get_recommendations_based_on_entry(918,[2904,28891],[30276],False)

In [57]:

# get recommendations based on user behavior not entry
async def get_global_recommendations(watched_anime_ids, favorite_anime_ids, return_entry=False):
  # get rid of the duplications
  anime_ids = set(watched_anime_ids + favorite_anime_ids)
  # get the clusters in the user lists
  anime_entries = anime_data[anime_data["anime_id"].isin(anime_ids)]
  cluster_ids = Counter(anime_entries["cluster"])

  # Get top N most common clusters (can be > n_clusters for sampling)
  top_clusters = [cluster for cluster, _ in cluster_ids.most_common(6)]
  selected_clusters = random.sample(top_clusters, k=min(top_n_clusters, len(top_clusters)))

  # get candidate anime
  candidate_anime = anime_data[anime_data["cluster"].isin(selected_clusters)]
  candidate_anime = candidate_anime[~candidate_anime["anime_id"].isin(anime_ids)]

  # some sorting
  recommended = candidate_anime.head(top_n_anime)
  return await get_anime_list(recommended) if return_entry else recommended

In [ ]:
await get_global_recommendations([28977,9969,15335,15417,918],[30276,245,21], True)

In [60]:
# get the cold start recommendations
async def get_cold_start(most_popular_clusters=5, k_per_cluster=5, return_entry=False):
    # Get top N most popular clusters by average member count
    popular_clusters = (
        anime_data.groupby("cluster")["members"]
        .mean()
        .sort_values(ascending=False)
        .head(most_popular_clusters)
        .index
    )

    # Collect top-k anime from each cluster
    clusters_anime_list = []
    for cluster_id in popular_clusters:
        cluster_anime = anime_data[anime_data["cluster"] == cluster_id]
        top_k_anime = cluster_anime.sort_values(by="members", ascending=False).head(k_per_cluster)
        clusters_anime_list.append(top_k_anime)

    # Concatenate all into one DataFrame
    recommended = pd.concat(clusters_anime_list).reset_index(drop=True)
    return await get_anime_list(recommended) if return_entry else recommended


In [ ]:
await get_cold_start(most_popular_clusters,k_per_cluster, True)

In [22]:
def plot_clusters_2d(feature_vectors, labels, sample_size=1000):
    """
    Plots anime clusters in 2D using PCA.
    
    Args:
        feature_vectors: np.array of all features (already normalized).
        labels: Cluster labels (e.g., anime_data["cluster"])
        sample_size: Limit for faster plotting (optional, set to None for all)
    """
    # Sample data if needed to avoid clutter
    if sample_size is not None and sample_size < len(feature_vectors):
        np.random.seed(42)
        indices = np.random.choice(len(feature_vectors), sample_size, replace=False)
        features_sample = feature_vectors[indices]
        labels_sample = np.array(labels)[indices]
    else:
        features_sample = feature_vectors
        labels_sample = labels

    # Reduce dimensions using PCA
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(features_sample)

    # Plot
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(reduced[:, 0], reduced[:, 1], c=labels_sample, cmap='tab20', alpha=0.6, s=30)
    plt.title("Anime Clusters Visualization (PCA)", fontsize=16)
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.colorbar(scatter, label='Cluster')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_clusters_2d(feature_vectors, anime_data["cluster"], sample_size=1000)